# Extract Rating data from the csv file generated by jsPsych

Use this notebook for the Rating task. It reads raw jsPsych CSV files and makes a cleaner table with image number and slider rating.

## What can I change?

At the bottom of this notebook, change the input folder or file name so it points to your own data.


In [1]:
import os
import re
import glob
import json
import numpy as np
import pandas as pd

In [2]:
def extract_image_idx(stimulus):
    if pd.isna(stimulus):
        return np.nan

    fname = os.path.basename(str(stimulus))
    match = re.search(r"(\d+)", fname)

    if match:
        return int(match.group(1))

    return np.nan


def parse_survey_response(response):
    if pd.isna(response):
        return {}

    try:
        return json.loads(response)
    except:
        return {}


def age_to_midpoint(age_str):
    if pd.isna(age_str):
        return np.nan

    age_str = str(age_str)

    if "-" in age_str:
        lo, hi = age_str.split("-")
        return (float(lo) + float(hi)) / 2

    if "+" in age_str:
        return float(age_str.replace("+", ""))

    return float(age_str)


def extract_ratings_file(csv_file):
    subject_id = os.path.splitext(os.path.basename(csv_file))[0].split("_")[-1]
    df = pd.read_csv(csv_file)

    trials = df[
        (df["trial_type"] == "html-slider-response")
        & df["stimulus"].notna()
    ].copy()

    stimuli = df[(df["trial_type"] == "image-button-response") & df["stimulus"].notna()]
    stimuli = stimuli.reset_index(drop=True)
    trials = trials.reset_index(drop=True)

    trials["image_idx"] = stimuli["stimulus"].apply(extract_image_idx).values
    trials["stimulus"] = stimuli["stimulus"].values

    trials["rating"] = pd.to_numeric(
        trials["response"],
        errors="coerce", downcast='integer'
    )
    trials["slider_start"] = pd.to_numeric(
        trials["slider_start"],
        errors="coerce", downcast='integer'
    )

    trials["subject_id"] = subject_id
    trials["source_file"] = os.path.basename(csv_file)

    out = trials[
        [
            "source_file",
            "subject_id",
            "rt",
            "stimulus",
            "image_idx",
            "slider_start",
            "rating"
        ]
    ].copy()

    return out

In [3]:
def extract_ratings_folder(
    input_folder,
    output_csv="ratings_extracted.csv"
):

    all_files = glob.glob(os.path.join(input_folder, "*.csv"))

    all_data = []

    ages = []
    sexes = []

    excluded = 0
    for f in all_files:
        if "extracted" in f:
          excluded += 1
          continue

        raw_df = pd.read_csv(f)

        survey_rows = raw_df[
            raw_df["trial_type"] == "survey-multi-choice"
        ]

        if len(survey_rows) > 0:

            survey_response = survey_rows.iloc[0]["response"]
            survey = parse_survey_response(survey_response)

            if "Age" in survey:
                ages.append(survey["Age"])

            if "Gender" in survey:
                sexes.append(survey["Gender"])

        parsed = extract_ratings_file(f)
        all_data.append(parsed)

    results = pd.concat(all_data, ignore_index=True)
    results.to_csv(output_csv, index=False)

    # ---------------------------------
    # Print survey statistics
    # ---------------------------------
    print("\n==============================")
    print("Survey Statistics")

    print(f"N participants: {len(all_files)-excluded}")

    if len(ages) > 0:
        age_counts = pd.Series(ages).value_counts()
        print("\nAge distribution:")
        for age, count in age_counts.items():
            print(f"  {age}: {count}")

    if len(sexes) > 0:
        sex_counts = pd.Series(sexes).value_counts()

        print("\nGender distribution:")
        for sex, count in sex_counts.items():
            print(f"  {sex}: {count}")

    # -----------------------------------
    # Task statistics
    # -----------------------------------
    print("\n==============================")
    print("Ratings Statistics")

    print(f"N trials: {len(results)}")
    print(f"Mean rating: {results['rating'].mean():.3f}")
    print(f"Std rating: {results['rating'].std():.3f}")
    print(f"Mean RT: {results['rt'].mean():.2f} ms")

    print("\nSaved parsed data to:")
    print(output_csv)

    return results

In [ ]:
# ===============================
# STUDENTS: EDIT THIS PART BELOW
# ===============================

In [4]:
# Example: parse one file
results = extract_ratings_file("./data/ratings/ratings_481520.csv")
results.to_csv("./data/ratings/ratings_481520_extracted.csv", index=False)

In [5]:
# Example: parse all files in a folder
results = extract_ratings_folder("./data/ratings","./data/all_data_ratings.csv")


Survey Statistics
N participants: 1

Age distribution:
  26-30: 1

Gender distribution:
  Female: 1

Ratings Statistics
N trials: 5
Mean rating: 2.800
Std rating: 0.837
Mean RT: 2216.56 ms

Saved parsed data to:
./data/all_data_ratings.csv
